# 09 -- Pattern detector validation

Paired script: `analysis/pattern_validation.py` -- Python ports of
`CandlestickPatternEngine.mqh`'s pattern predicates, kept algebraically identical to the
MQL5 source.

**Fixed, 2026-07-21 Codex review finding:** this notebook previously checked only bullish
engulfing in a 3-bar array; with the function's default `trend_lookback=5`, neither pin-bar
predicate could even be evaluated (the fixture was too short), despite a comment claiming an
embedded bullish pin bar. It also never invoked `compare_to_mql5_export`. Both are fixed here:
the pin-bar fixture is now long enough for the default `trend_lookback`, and the MQL5
comparison helper is invoked (against a self-consistent synthetic export -- see the closing
cell for what a real comparison still requires).

**Corrected, 2026-07-22 Codex review finding (eighth round, P1 finding 17):** the text above
and below previously understated this module's own scope ("first slice", "only 4 ... are
ported") and claimed no MQL5 exporter exists at all -- both stale. TASK-033 completed all 20
of `CandlestickPatternEngine.mqh`'s detector/predicate functions in `pattern_validation.py`
(see that module's own docstring), and `Export_PatternDetectorResults.mq5` (TASK-037) now
exports a real MQL5 detector-results CSV, including symbol/timestamp/OHLC/ATR provenance
columns specifically so a comparison can PROVE both sides analyzed the same underlying bars,
not merely the same row count (see `compare_to_mql5_export`'s own docstring for the exact
counterexample this closes).

In [1]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.pattern_validation import compare_to_mql5_export, detect_all_patterns

In [2]:
# Bullish engulfing at k=0 (same 2-bar fixture as tests/test_pattern_validation.py).
engulfing_result = detect_all_patterns(
    opens=[99.0, 110.0],
    highs=[113.0, 111.0],
    lows=[98.0, 99.0],
    closes=[112.0, 100.0],
)
print("Engulfing fixture:")
print(engulfing_result)
assert bool(engulfing_result.iloc[0]["bullish_engulfing"]) is True

Engulfing fixture:
   k  bullish_pin_bar  bearish_pin_bar  bullish_engulfing  bearish_engulfing  \
0  0            False            False               True              False   
1  1            False            False              False              False   

   dragonfly_rejection  gravestone_rejection   doji  spinning_top  inside_bar  \
0                False                 False  False         False       False   
1                False                 False  False         False       False   

   outside_bar  harami_detected  harami_confirmed  morning_star  evening_star  \
0         True            False             False         False         False   
1        False            False             False         False         False   

   three_white_soldiers  three_black_crows  
0                 False              False  
1                 False              False  


In [3]:
# Bullish pin bar at k=0 with the DEFAULT trend_lookback=5 -- needs 6 bars
# (index 0..5), unlike the previous (broken) 3-bar version of this
# notebook. Bar 0 is the same hand-verified pin-bar shape as
# tests/test_pattern_validation.py::test_bullish_pin_bar_detected; bars
# 1-4 are irrelevant filler (only bar 0 and bar 5 affect the result);
# bar 5's close (125) > bar 0's close (118) satisfies the "preceding
# down-move" condition at trend_lookback=5.
pin_bar_result = detect_all_patterns(
    opens=[116.0, 100.0, 100.0, 100.0, 100.0, 124.0],
    highs=[120.0, 101.0, 101.0, 101.0, 101.0, 126.0],
    lows=[100.0, 99.0, 99.0, 99.0, 99.0, 123.0],
    closes=[118.0, 100.0, 100.0, 100.0, 100.0, 125.0],
    trend_lookback=5,
)
print("Pin-bar fixture:")
print(pin_bar_result)
assert bool(pin_bar_result.iloc[0]["bullish_pin_bar"]) is True

Pin-bar fixture:
   k  bullish_pin_bar  bearish_pin_bar  bullish_engulfing  bearish_engulfing  \
0  0             True            False              False              False   
1  1            False            False              False              False   
2  2            False            False              False              False   
3  3            False            False              False              False   
4  4            False            False              False              False   
5  5            False            False              False              False   

   dragonfly_rejection  gravestone_rejection   doji  spinning_top  inside_bar  \
0                 True                 False   True         False       False   
1                False                 False   True         False       False   
2                False                 False   True         False       False   
3                False                 False   True         False       False   
4                

## Cross-check against an MQL5 export

`compare_to_mql5_export` is invoked here against a SELF-CONSISTENT synthetic export -- the
Python results themselves, re-saved to CSV with matching OHLC identity columns -- which proves
the comparison MECHANISM works (zero disagreements against itself, including the identity
check) without claiming this is a real cross-check against MQL5. A real comparison requires
actually running `Export_PatternDetectorResults.mq5` against a live/demo MT5 terminal (this
sandbox cannot attach to one -- see the closing cell) and pointing this notebook at that real
export file instead.

In [4]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_pattern_demo_"))
synthetic_mql5_export = tmp_dir / "mql5_export.csv"

# **Extended, 2026-07-22 Codex review finding (eighth round, P1 finding 17):**
# compare_to_mql5_export now REQUIRES python_identity (the bar's own OHLC, at minimum) to
# prove both sides analyzed the same underlying bars, not just the same row count -- the
# same identity columns are added to the synthetic export here so the self-comparison below
# still exercises the identity check itself (matching, since this is a self-consistent
# export), not just the pattern-boolean comparison.
engulfing_identity = pd.DataFrame(
    {
        "k": [0, 1],
        "open": [99.0, 110.0],
        "high": [113.0, 111.0],
        "low": [98.0, 99.0],
        "close": [112.0, 100.0],
    }
)
export_df = engulfing_result.copy()
for col in ["open", "high", "low", "close"]:
    export_df[col] = engulfing_identity[col]
export_df.to_csv(synthetic_mql5_export, index=False)

disagreements = compare_to_mql5_export(
    engulfing_result, synthetic_mql5_export, python_identity=engulfing_identity
)
print(f"disagreements vs. self-consistent synthetic export: {len(disagreements)}")
assert len(disagreements) == 0

disagreements vs. self-consistent synthetic export: 0


## Cross-check against a REAL MQL5 detector export: batched runtime verification

The self-comparison above only proves `compare_to_mql5_export`'s join/diff/identity logic is
correct, not that the Python and MQL5 detectors agree on real data. `Export_PatternDetectorResults.mq5`
(TASK-037) already exists and exports a real detector-results CSV (including the symbol/
timestamp/OHLC/ATR provenance columns `compare_to_mql5_export`'s own identity check now
verifies) -- running it against a live/demo MT5 terminal and pointing this notebook at that
real export file remains part of this project's batched runtime-verification backlog (this
sandbox cannot attach to a live/demo terminal). All 20 of `CandlestickPatternEngine.mqh`'s
detector/predicate functions are ported in `pattern_validation.py` (see that module's own
docstring), so a real comparison run is limited only by MT5 terminal access, not by remaining
Python-side port work.